# 01 - Data preparation and pronunciation-code discovery

All the offline work happens here. Afterwards training reads only memmaps, so
the GPU never waits on data.

| Stage | What it does | Time on a 4090 |
|---|---|---|
| A0 | manifest: scan, normalize, filter | 1 min |
| A1 | CTC forced alignment to word spans | about 35 min |
| A2 | self-supervised embeddings per word span | about 25 min |
| A3 | **discover pronunciation codes** | about 8 min |
| A4 | MARBERTv2 teacher cache and head | about 8 min |
| A5 | Mimi encode to RVQ codes | about 30 min |

Stage A3 is the novel part: it decides from audio alone which words have more
than one pronunciation. Nothing is hardcoded.

In [ ]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.environ["PYTHONIOENCODING"] = "utf-8"

# ---------------------------------------------------------------------------
# PICK YOUR EXPERIMENT HERE. This is the only line to change.
#
#   configs/exp0_small.yaml     2013 clips, ~2 GB   -> proves the pipeline,
#                                                     runs on a 6 GB GPU
#   configs/exp1_egyptian.yaml  15.6k clips, 68 h   -> the real run
# ---------------------------------------------------------------------------
CONFIG = "configs/exp0_small.yaml"

from adaptts.utils.config import load_config
from adaptts.utils.logging_utils import setup_logging
setup_logging()
cfg = load_config(CONFIG)
print("repo   :", REPO)
print("config :", CONFIG, "->", cfg.name)
print("dataset:", cfg.paths.hf_dataset_id)

# The homographs named in the brief, read from the probe file so the notebooks
# never hardcode a word list of their own.
import json as _json
PROBE_WORDS = sorted({
    w for _s in _json.load(open("assets/probe_sentences.json", encoding="utf-8"))["sentences"]
    for w in _s["focus"].split(" / ") if w and w != "none"
})
print("probe  :", " ".join(PROBE_WORDS))

## Download the dataset

About 12 GB. To use a different corpus, change `paths.hf_dataset_id` and
`paths.dataset_dir` in the config.

In [ ]:
from huggingface_hub import snapshot_download

target = cfg.paths.dataset_dir
print("downloading to:", target)
snapshot_download(
    repo_id=cfg.paths.hf_dataset_id, repo_type="dataset",
    local_dir=target, max_workers=8,
)
print("done")

In [ ]:
import glob, os

wavs = glob.glob(os.path.join(target, "clips", "**", "*.wav"), recursive=True)
parquet = glob.glob(os.path.join(target, "**", "*.parquet"), recursive=True)
meta = os.path.join(target, "metadata")
print("wav clips     :", len(wavs))
print("parquet shards:", len(parquet))
print("metadata dir  :", os.listdir(meta) if os.path.isdir(meta) else "none")
if parquet and not wavs:
    print()
    print("This is a parquet dataset. The manifest stage below unpacks the")
    print("embedded audio to wav once, so later stages never decode it again.")

## Stage A0 - manifest

Normalizes every transcript once, so alignment, the teacher and training all see
byte-identical strings.

In [ ]:
!python scripts/preprocess.py --config $CONFIG --stage manifest

In [ ]:
import json, collections

rows = [json.loads(l) for l in open(cfg.paths.manifest_path, encoding="utf-8")]
hours = sum(r["duration"] for r in rows) / 3600
print(f"{len(rows)} utterances, {hours:.1f} hours")
print("splits:", collections.Counter(r["split"] for r in rows))
for r in rows[:3]:
    print(f'  [{r["duration"]:.1f}s] {r["text"][:70]}')

### Check the text normalization

Every transcript passed through the Egyptian normalizer. Numbers, dates and
Latin tokens should all be spoken words by now, with no digits left.

In [ ]:
import json, random

rows = [json.loads(l) for l in open(cfg.paths.manifest_path, encoding="utf-8")]
random.seed(0)
for r in random.sample(rows, min(8, len(rows))):
    print(f"[{r['duration']:5.1f}s] {r['text'][:100]}")

leftover = [r for r in rows if any(c.isdigit() for c in r["text"])]
print()
print(f"utterances still containing digits: {len(leftover)} of {len(rows)}")
for r in leftover[:3]:
    print("   ", r["text"][:100])

## Stage A1 - CTC forced alignment

Finds the time span of every word with no pronunciation lexicon. The Viterbi
alignment is implemented in-repo, so there is no Montreal Forced Aligner
dependency.

In [ ]:
!python scripts/preprocess.py --config $CONFIG --stage align

## Stages A2 and A3 - span embeddings and code discovery

The heart of the system. Each occurrence of each word type is embedded with a
self-supervised speech model, then we ask whether those embeddings form one
cluster or several. A split is accepted only when it is reproducible under
bootstrap resampling, geometrically clean, and acoustically well separated.

In [ ]:
!python scripts/preprocess.py --config $CONFIG --stage discover

### What did it discover?

This is the moment of truth. Words like the ones named in the brief should
appear with more than one reading.

In [ ]:
from adaptts.data.discovery import PronunciationLexicon

lex = PronunciationLexicon.load(cfg.paths.lexicon_path)
amb = lex.ambiguous_words
print(f"{len(amb)} ambiguous word types discovered")
print()

entries = sorted((lex.entries[w] for w in amb), key=lambda e: -e.occurrence_count)
header = f"{'word':<18}{'codes':>6}{'occ':>7}{'stab':>8}{'sil':>8}{'sep':>8}  counts"
print(header)
print("-" * 74)
for e in entries[:40]:
    print(f"{e.word:<18}{e.n_codes:>6}{e.occurrence_count:>7}{e.stability:>8.2f}"
          f"{e.silhouette:>8.2f}{e.separation:>8.2f}  {e.counts}")

In [ ]:
# Check the specific homographs named in the project brief.
for w in PROBE_WORDS:
    k = lex.n_codes(w)
    print(f"{w:<10} -> {k} code(s)   " + ("AMBIGUOUS" if k > 1 else "single reading"))

In [ ]:
# Read the sentences behind each code. This is how you confirm the clusters
# track meaning rather than recording conditions.
import json, collections, os

labels = json.load(open(os.path.join(cfg.paths.cache_dir, "code_labels.json"), encoding="utf-8"))
manifest = [json.loads(l) for l in open(cfg.paths.manifest_path, encoding="utf-8")]
by_uid = {r["uid"]: r["text"] for r in manifest}

WORD = PROBE_WORDS[0]      # change to inspect any discovered homograph
groups = collections.defaultdict(list)
for key, code in labels.items():
    uid, widx = key.split(chr(9))
    text = by_uid.get(uid, "")
    words = text.split()
    if int(widx) < len(words) and words[int(widx)] == WORD:
        groups[code].append(text)

for code in sorted(groups):
    print()
    print(f"=== {WORD}  code {code}  ({len(groups[code])} occurrences) ===")
    for t in groups[code][:6]:
        print("   ", t[:95])

If the sentences under each code share a meaning, discovery worked. If they look
mixed, raise `discovery.min_separation` or `discovery.stability_threshold` in the
config and rerun this stage with `--force`.

## Stage A4 - teacher cache

One frozen MARBERTv2 pass over the corpus, cached as fp16. The teacher never
runs again, which is the main reason training is cheap.

In [ ]:
!python scripts/preprocess.py --config $CONFIG --stage teacher

## Stage A5 - Mimi codec encoding

Every clip becomes 8 RVQ streams at 12.5 Hz. A 10 second clip is 125 frames,
which is why generation is fast on a CPU.

In [ ]:
!python scripts/preprocess.py --config $CONFIG --stage codec

## Verify the cache is complete

In [ ]:
from adaptts.data.dataset import AdapTTSDataset, collate
from adaptts.text.vocab import CharVocab

vocab = CharVocab.load(cfg.paths.charvocab_path)
ds = AdapTTSDataset(cfg, "train", vocab, lex, need_codes=True, need_teacher=True)
b = collate(
    [ds[i] for i in range(4)], vocab.pad_id, cfg.discovery.max_codes_per_word,
    cfg.audio.n_quantizers, cfg.teacher.hidden_size,
)
for k, v in b.items():
    print(f"  {k:<20} {tuple(v.shape)}  {v.dtype}")
print()
print("ambiguous words in this batch:", int((b["n_codes"] > 1).sum()))
ds.close()
print()
print("Data is ready. Continue to 02_train_context.ipynb")